In [1]:
import numpy as np
import matplotlib.pyplot as plt
from mpdaf.obj import Cube
from astropy.coordinates import SkyCoord
from astropy.wcs import WCS
import sys
import os
import plotfancy as pf
from matplotlib.patches import Circle
from astropy.visualization import ZScaleInterval
from astropy.table import Table
from types import SimpleNamespace
import re
from astropy.io import ascii
from astropy import units as u
from astropy.table import Table, vstack, hstack
from astropy.modeling import models, fitting

import os
from io import StringIO
from astropy.table import vstack

# from calculate_jiang19_metallicity import calculate_metallicity_jiang19 as cjm19
sys.path.append('../../../')
import src.ifu_tools.line_ratios as lr
# import src.ifu_tools.ifutools as ift
from src.ifu_tools.ifutools import museCube
# import line_ratios as lr

import logging 
logging.getLogger('mpdaf').setLevel(logging.WARNING)

import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

from astropy.table import join
from astroquery.sdss import SDSS

pf.housestyle_rcparams()

True

In [2]:
class QT_Candidates:
    def __init__(self, file_path: str = 'leadlines.csv'):
        self.file_path = file_path
        # self._data = {}
        self._keys = []
        self.analysed = False # ticker for coadd 
        self._initialise_file()
    
    def _initialise_file(self):
        self._leadlines = ascii.read('leadlines.csv')
        kyz = []
        for k in self._leadlines:
            kyz.append((k['dir'],k['key']))
        
        # Get unique (dir, key) tuples and sort them to ensure a consistent order.
        self._unique_kyz = sorted(list(set(kyz)))
        
        # Derive the simple keys from the sorted, unique list.
        self._keys = [k for (d,k) in self._unique_kyz]

        self._leadlines_OIII = self._leadlines[self._leadlines['Redshift']<0.8]

    def keys(self):
        return self._keys
    
    def get_candidate(self, key: str):
        global cand_leadlines
        cand_leadlines = self._leadlines_OIII[self._leadlines_OIII['key'] == key]

        key = cand_leadlines['key'][0]
        dir = cand_leadlines['dir'][0]
        loc = '/Volumes/Expansion/exp_thardy/'+dir+'/'+key+'_COMBINED_CUBE_MED_FINAL.fits'

        # with fits.open(loc) as hdul:
        #     hdr = hdul[0].header
        print(loc)
        hdr = Cube(loc).get_wcs_header()
        
        w = WCS(hdr)

        coords,wls = w.pixel_to_world(cand_leadlines['X_PEAK_SN'],
                                cand_leadlines['Y_PEAK_SN'],
                                cand_leadlines['Z_PEAK_SN'])
        
        cand_leadlines['ra'] = coords.ra.deg
        cand_leadlines['dec'] = coords.dec.deg
        cand_leadlines['OIII_est'] = (cand_leadlines['Redshift']+1)*5007

        crvals = hdr['CRVAL1'], hdr['CRVAL2']

        return cand_leadlines, cand_leadlines['zcluster'][0], crvals
        # redshift, table

    def analyse_all(self, raw=False, save_extras=False):
        self.analysed = True

        # Define output filenames
        output_filename = 'allsources.csv'
        raw_output_filename = 'allsources_uncorrected.csv'

        # Ensure a clean start by removing old files before the run begins.
        if os.path.exists(output_filename):
            os.remove(output_filename)
        if raw and os.path.exists(raw_output_filename):
            os.remove(raw_output_filename)

        # Store spectra and tables in memory to build final attributes
        spectra = {}
        all_tables = []
        all_raw_tables = []

        # Iterate over the unified list of unique (dir, key) tuples to avoid mismatches.
        for (d, k) in self._unique_kyz:
            name = k # This is the simple key name
            path = f'/Volumes/Expansion/exp_thardy/{d}/{k}_COMBINED_CUBE_MED_FINAL.fits'
            
            print(f'running {name}')
            try:
                tab, z, crvals = self.get_candidate(name)

                # Use the correct path that corresponds to the candidate's data (tab).
                indiv_cube = museCube(path, cluster_ra=crvals[0], cluster_dec=crvals[1])
                indiv_cube.process_multiple_candidates(tab, zcl=z)

                # --- Appending logic for the main table ---
                ex_table = indiv_cube.ex_table
                all_tables.append(ex_table)
                
                write_header = not os.path.exists(output_filename)
                s_buf = StringIO()
                ex_table.write(s_buf, format='csv')
                s_buf.seek(0)
                lines = s_buf.readlines()
                
                with open(output_filename, 'a') as f:
                    if write_header:
                        f.write(lines[0])
                    f.writelines(lines[1:])

                # --- Appending logic for the raw table (if requested) ---
                if raw:
                    raw_table = indiv_cube.raw_table
                    all_raw_tables.append(raw_table)
                    
                    write_header_raw = not os.path.exists(raw_output_filename)
                    s_buf_raw = StringIO()
                    raw_table.write(s_buf_raw, format='csv')
                    s_buf_raw.seek(0)
                    lines_raw = s_buf_raw.readlines()
                    
                    with open(raw_output_filename, 'a') as f_raw:
                        if write_header_raw:
                            f_raw.write(lines_raw[0])
                        f_raw.writelines(lines_raw[1:])
                
                if save_extras:
                    spectra[name] = indiv_cube.rest_spectra
            except:
                print('Failed, Skipping')

        # Create the combined table attributes from the in-memory lists
        if save_extras:
            self.combined_table = vstack(all_tables)
            if raw:
                self.combined_table_raw = vstack(all_raw_tables)

            self.spectra = spectra      

In [3]:
# class QT_Candidates:
#     def __init__(self, file_path: str = 'leadlines.csv'):
#         self.file_path = file_path
#         # self._data = {}
#         self._keys = []
#         self.analysed = False # ticker for coadd 
#         self._initialise_file()
    
#     def _initialise_file(self):
#         self._leadlines = ascii.read('leadlines.csv')
#         kyz = []
#         for k in self._leadlines:
#             kyz.append((k['dir'],k['key']))
        
#         # Get unique (dir, key) tuples and sort them to ensure a consistent order.
#         self._unique_kyz = sorted(list(set(kyz)))
        
#         # Derive the simple keys from the sorted, unique list.
#         self._keys = [k for (d,k) in self._unique_kyz]

#         self._leadlines_OIII = self._leadlines[self._leadlines['Redshift']<0.8]

#     def keys(self):
#         return self._keys
    
#     def get_candidate(self, key: str):
#         global cand_leadlines
#         cand_leadlines = self._leadlines_OIII[self._leadlines_OIII['key'] == key]

#         key = cand_leadlines['key'][0]
#         dir = cand_leadlines['dir'][0]
#         loc = '/Volumes/Expansion/exp_thardy/'+dir+'/'+key+'_COMBINED_CUBE_MED_FINAL.fits'

#         # with fits.open(loc) as hdul:
#         #     hdr = hdul[0].header
#         print(loc)
#         hdr = Cube(loc).get_wcs_header()
        
#         w = WCS(hdr)

#         coords,wls = w.pixel_to_world(cand_leadlines['X_PEAK_SN'],
#                                 cand_leadlines['Y_PEAK_SN'],
#                                 cand_leadlines['Z_PEAK_SN'])
        
#         cand_leadlines['ra'] = coords.ra.deg
#         cand_leadlines['dec'] = coords.dec.deg
#         cand_leadlines['OIII_est'] = (cand_leadlines['Redshift']+1)*5007

#         crvals = hdr['CRVAL1'], hdr['CRVAL2']

#         return cand_leadlines, cand_leadlines['zcluster'][0], crvals
#         # redshift, table

#     def analyse_all(self, raw=False):
#         self.analysed = True

#         tables = {}
#         raw_tables = {}
#         spectra = {}
#         # Iterate over the unified list of unique (dir, key) tuples to avoid mismatches.
#         for (d, k) in self._unique_kyz:
#             name = k # This is the simple key name
#             path = f'/Volumes/Expansion/exp_thardy/{d}/{k}_COMBINED_CUBE_MED_FINAL.fits'
            
#             print(f'running {name}')
#             global tab,z,crvals
#             tab, z, crvals = self.get_candidate(name)

#             # Use the correct path that corresponds to the candidate's data (tab).
#             indiv_cube = museCube(path, cluster_ra=crvals[0], cluster_dec=crvals[1])
#             indiv_cube.process_multiple_candidates(tab, zcl=z)

#             tables[name] = indiv_cube.ex_table
#             raw_tables[name] = indiv_cube.raw_table
#             spectra[name] = indiv_cube.rest_spectra

#         self.combined_table = vstack(list(tables.values()))
#         self.combined_table.write('allsources.csv', overwrite=True)

#         if raw:
#             self.combined_table_raw = vstack(list(raw_tables.values()))
#             self.combined_table_raw.write('allsources_uncorrected.csv', overwrite=True)

#         self.spectra = spectra

In [4]:
cand = QT_Candidates()
cand.analyse_all()

running a141
/Volumes/Expansion/exp_thardy/cubes/a141_COMBINED_CUBE_MED_FINAL.fits


running a209
/Volumes/Expansion/exp_thardy/cubes/a209_COMBINED_CUBE_MED_FINAL.fits
running a2697
/Volumes/Expansion/exp_thardy/cubes/a2697_COMBINED_CUBE_MED_FINAL.fits
running a2811
Failed, Skipping
running a2813
/Volumes/Expansion/exp_thardy/cubes/a2813_COMBINED_CUBE_MED_FINAL.fits
running a3017
/Volumes/Expansion/exp_thardy/cubes/a3017_COMBINED_CUBE_MED_FINAL.fits
running macs0011m15
/Volumes/Expansion/exp_thardy/cubes/macs0011m15_COMBINED_CUBE_MED_FINAL.fits
running macs0018m40
/Volumes/Expansion/exp_thardy/cubes/macs0018m40_COMBINED_CUBE_MED_FINAL.fits


running macs0027p26a
Failed, Skipping
running macs0027p26b
Failed, Skipping
running macs0028m75
/Volumes/Expansion/exp_thardy/cubes/macs0028m75_COMBINED_CUBE_MED_FINAL.fits
running macs0032p18
/Volumes/Expansion/exp_thardy/cubes/macs0032p18_COMBINED_CUBE_MED_FINAL.fits
running macs0033m07
/Volumes/Expansion/exp_thardy/cubes/macs0033m07_COMBINED_CUBE_MED_FINAL.fits
running macs0034p02a
/Volumes/Expansion/exp_thardy/cubes/macs0034p02a_COMBINED_CUBE_MED_FINAL.fits
running macs0034p02b
/Volumes/Expansion/exp_thardy/cubes/macs0034p02b_COMBINED_CUBE_MED_FINAL.fits
running macs0040m44
Failed, Skipping
running macs0051p27
Failed, Skipping
running macs0111p08
Failed, Skipping
running macs0138m21
/Volumes/Expansion/exp_thardy/cubes/macs0138m21_COMBINED_CUBE_MED_FINAL.fits
running macs0140m05
Failed, Skipping
running macs0140m34
Failed, Skipping
running macs0152m28
/Volumes/Expansion/exp_thardy/cubes/macs0152m28_COMBINED_CUBE_MED_FINAL.fits
running macs0159m34
/Volumes/Expansion/exp_thardy/cubes/

running rxj0218m31
/Volumes/Expansion/exp_thardy/cubes/rxj0218m31_COMBINED_CUBE_MED_FINAL.fits
running rxj0232m44
/Volumes/Expansion/exp_thardy/cubes/rxj0232m44_COMBINED_CUBE_MED_FINAL.fits
running s26
/Volumes/Expansion/exp_thardy/cubes/s26_COMBINED_CUBE_MED_FINAL.fits
running s780
/Volumes/Expansion/exp_thardy/cubes/s780_COMBINED_CUBE_MED_FINAL.fits
running a3186
/Volumes/Expansion/exp_thardy/cubes_new/a3186_COMBINED_CUBE_MED_FINAL.fits
running a3322
/Volumes/Expansion/exp_thardy/cubes_new/a3322_COMBINED_CUBE_MED_FINAL.fits
running a3364
/Volumes/Expansion/exp_thardy/cubes_new/a3364_COMBINED_CUBE_MED_FINAL.fits
running a3378
/Volumes/Expansion/exp_thardy/cubes_new/a3378_COMBINED_CUBE_MED_FINAL.fits
running a3399
/Volumes/Expansion/exp_thardy/cubes_new/a3399_COMBINED_CUBE_MED_FINAL.fits
running a383
/Volumes/Expansion/exp_thardy/cubes_new/a383_COMBINED_CUBE_MED_FINAL.fits
running a520
Failed, Skipping
running macs0035m20
Failed, Skipping
running macs0238m53
/Volumes/Expansion/exp_thar

running macs0406m49
Failed, Skipping
running macs0449m28
/Volumes/Expansion/exp_thardy/cubes_new/macs0449m28_COMBINED_CUBE_MED_FINAL.fits
fit for BLANK OIII line at 4344.862951236708 failed, using guess z
running macs0455p06
/Volumes/Expansion/exp_thardy/cubes_new/macs0455p06_COMBINED_CUBE_MED_FINAL.fits
running macs0529m31
/Volumes/Expansion/exp_thardy/cubes_new/macs0529m31_COMBINED_CUBE_MED_FINAL.fits
running macs0547m39
/Volumes/Expansion/exp_thardy/cubes_new/macs0547m39_COMBINED_CUBE_MED_FINAL.fits
running macs0553m33a
Failed, Skipping
running macs0553m33b
/Volumes/Expansion/exp_thardy/cubes_new/macs0553m33b_COMBINED_CUBE_MED_FINAL.fits
running macs0600m20
/Volumes/Expansion/exp_thardy/cubes_new/macs0600m20_COMBINED_CUBE_MED_FINAL.fits
running macs0611m30
/Volumes/Expansion/exp_thardy/cubes_new/macs0611m30_COMBINED_CUBE_MED_FINAL.fits
running macs0723m73
/Volumes/Expansion/exp_thardy/cubes_new/macs0723m73_COMBINED_CUBE_MED_FINAL.fits
running rbs653
/Volumes/Expansion/exp_thardy/cub

In [5]:
cand_leadlines

I,ID,X_PEAK_SN,Y_PEAK_SN,Z_PEAK_SN,DETSN_MAX,Identification,Short,Comment,Redshift,Quality,Confidence,Association,key,cluster,zcluster,dir,ra,dec,OIII_est
int64,int64,float64,float64,float64,float64,str6,str5,str2,float64,str1,int64,str1,str12,str17,float64,str9,float64,float64,float64
8834,117,172.0,161.0,1104.0,60.877079010009766,OII,O2,-,0.64387,c,0,-,z2089,Zw2089,nan,cubes_new,135.15351058267362,20.89469129914648,8230.857090000001
9118,122,255.0,231.0,1876.0,40.883541107177734,Ha,Ha,-,0.08105,c,0,-,z2089,Zw2089,nan,cubes_new,135.1485748313986,20.898580196052816,5412.81735
9126,123,237.0,266.0,533.0,19.323928833007812,OIII 2,O3_2,-,0.08172,a,3,-,z2089,Zw2089,nan,cubes_new,135.14964523626062,20.900524650747624,5416.17204
9284,185,153.0,239.0,2654.0,18.397024154663086,Ha,Ha,-,0.22924,a,3,-,z2089,Zw2089,nan,cubes_new,135.15464055243186,20.899024610664153,6154.804679999999
9407,247,264.0,222.0,1876.0,9.173638343811035,Ha,Ha,-,0.08105,c,0,-,z2089,Zw2089,nan,cubes_new,135.1480396308023,20.898080188417477,5412.81735
9408,248,262.0,245.0,1877.0,10.07020378112793,Ha,Ha,-,0.08124,c,0,-,z2089,Zw2089,nan,cubes_new,135.14815854372222,20.899357968017405,5413.76868
9409,249,254.0,248.0,1877.0,8.544206619262695,Ha,Ha,-,0.08124,c,0,-,z2089,Zw2089,nan,cubes_new,135.14863428574972,20.899524641227718,5413.76868
9414,251,242.0,252.0,1879.0,9.175813674926758,Ha,Ha,-,0.08163,a,3,-,z2089,Zw2089,nan,cubes_new,135.1493479010342,20.89974687079747,5415.72141
9416,252,246.0,263.0,1879.0,8.462374687194824,Ha,Ha,-,0.08163,a,3,-,z2089,Zw2089,nan,cubes_new,135.14911002149546,20.9003579797741,5415.72141
